# V94 - IndoBERT encoder screen

Pre-registered plan: all tested encoders are multilingual or English, but the chats are Indonesian social-style text, exactly what IndoBERT pre-trained on. CLS vectors feed the same Ridge and cosine readouts under grouped folds with a fresh seed. Same locks. If Indonesian pre-training matters anywhere, it shows here.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 1919
N_SPLITS = 5
GATE_FLOOR = 0.002
MODEL = "indobenchmark/indobert-base-p1"
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v94")
OUT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "v94 needs CUDA"
print("DATA:", DATA.resolve())

## IndoBERT CLS vectors, then gate

CLS pooling per the BERT convention, L2 normalized. Same two readouts as every encoder screen.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
syll = [str(catalog.set_index("module_id").loc[m, "description_and_syllabus"]) if m in set(catalog["module_id"]) else "" for m in MODS]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

cu, cs = OUT / "ib_users.npy", OUT / "ib_syll.npy"
if cu.exists() and cs.exists():
    BU, BS = np.load(cu), np.load(cs)
    print("loaded cached vectors", BU.shape)
else:
    import torch.nn.functional as F
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL)
    net = AutoModel.from_pretrained(MODEL, torch_dtype=torch.float16, device_map={"": 0}).eval()

    def encode(texts, ml):
        out = np.zeros((len(texts), net.config.hidden_size), np.float32)
        with torch.no_grad():
            for b in range(0, len(texts), 16):
                e = tok(texts[b:b + 16], padding=True, truncation=True, max_length=ml, return_tensors="pt").to("cuda")
                h = net(**e).last_hidden_state
                v = h[:, 0, :].float()
                out[b:b + 16] = F.normalize(v, dim=-1).cpu().numpy()
        return out

    BU = encode(docs, 256)
    BS = encode(syll, 256)
    np.save(cu, BU)
    np.save(cs, BS)
    print("encoded", BU.shape)
    del net
    torch.cuda.empty_cache()
Bsim = BU[:n_train] @ BS.T
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_w, fid = ridge_oof(W[:n_train])
oof_b, _ = ridge_oof(BU[:n_train])
base = ndcg5(Y, oof_w)
for name, oo in [("indobert-ridge", oof_b), ("indobert-cosine", Bsim)]:
    s = ndcg5(Y, oo)
    d = s - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
    print(f"{name}: OOF={s.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if (lo > GATE_FLOOR and wf >= 0) else 'reject'}")